# Multiclass Classification with Logistic Regression
by Yesheng Guan, Yitian Liu, Yuchen Zhao
---

## Overview

This project tackles multiclass classification by adopting logistic regression as the model representation, which is decomposed into binary classification subtasks by two strategies: One-vs-All (OvA) and One-vs-One (OvO). Both strategies utilize cross-entropy loss to measure the divergence between predicted probabilities and true labels, while the Adam optimizer is employed to iteratively optimize model parameters.  

In OvA, $K$ binary classifiers (where $ K $ is the number of classes) are trained, each tasked with distinguishing one class from all others. In OvO, $ K(K-1)/2 $ classifiers are trained, with each focusing on a distinct pair of classes. For every classifier, Adam updates its parameters by combining momentum (to stabilize gradient updates) and adaptive learning rates (to adjust step sizes dynamically), minimizing cross-entropy loss on mini-batches iteratively.  

Advantages are: the interpretability of logistic regression; Adam’s robustness in converging rapidly even with noisy gradients; OvA’s simplicity and scalability to large $ K $; and OvO’s potential for more precise decision boundaries when class distributions are balanced.  

Disadvantages are: OvA’s inherent class imbalance problem (where “other classes” dominate in training); OvO’s high computational and memory overhead (scaling quadratically with $ K $); logistic regression’s limited capacity to model complex decision boundaries; and the need for careful hyperparameter tuning in Adam, particularly when dealing with OvO’s massive number of classifiers.

---

## Representation

### Mathematical Framework in One-vs-All (OvA)

In binary logistic regression, we model the probability of a single class using the sigmoid function. However, real-world problems often involve $K > 2$ classes. The One-vs-All (OvA) strategy, also known as One-vs-Rest (OvR), elegantly extends binary classification to handle multiple classes by decomposing the multiclass problem into $K$ independent binary classification subproblems.

The fundamental insight of OvA is that we can train $K$ separate binary classifiers, where each classifier $k$ learns to distinguish class $k$ from all other classes combined. This transformation allows us to leverage the well-understood binary logistic regression framework for more complex multiclass scenarios.


#### Input Feature Representation

Consider a dataset with $n$ training samples and $d$ features. Our input is represented as:

$$
\mathbf{X} \in \mathbb{R}^{n \times d}
$$

where each row $\mathbf{x}_i \in \mathbb{R}^d$ represents a single training example with $d$ features. For practical implementation, we often augment the feature matrix with a bias term by adding a column of ones, resulting in:

$$
\mathbf{X} \in \mathbb{R}^{n \times (d+1)}
$$

This allows us to incorporate the bias directly into the weight vector, simplifying our mathematical notation and implementation.

The target variable for multiclass classification is:

$$
\mathbf{y} \in \{1, 2, ..., K\}^n
$$

where $K$ is the total number of classes. Each $y_i$ indicates the class membership of the $i$-th training example.


#### Binary Classifier Construction for Each Class

For the One-vs-All approach, we construct $K$ binary classifiers. For each class $k \in \{1,2,...,K\}$, we create a binary classification problem by transforming the original labels:

$$
y_i^{(k)} =
\begin{cases}
1 & \text{if } y_i = k \\
0 & \text{if } y_i \neq k
\end{cases}
$$

This transformation creates $K$ different binary label vectors $\mathbf{y}^{(k)} \in \{0,1\}^n$, one for each classifier. Each binary classifier $k$ learns its own set of parameters:

- Weight vector: $\mathbf{w}_k \in \mathbb{R}^d$  
- Bias term: $b_k \in \mathbb{R}$

Alternatively, using the augmented feature representation, we have:

$$
\boldsymbol{\theta}_k =
\begin{bmatrix}
b_k \\
\mathbf{w}_k
\end{bmatrix}
\in \mathbb{R}^{d+1}
$$


#### The Sigmoid Function and Probability Estimation

Each binary classifier $k$ uses the logistic (sigmoid) function:

$$
f_k(\mathbf{x}) =
\sigma(\mathbf{w}_k^T \mathbf{x} + b_k)
=
\frac{1}{1 + e^{-(\mathbf{w}_k^T \mathbf{x} + b_k)}}
$$

where:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Here, $z_k = \mathbf{w}_k^T \mathbf{x} + b_k$ is the logit.


#### Interpreting the Classifier Outputs

The output $f_k(\mathbf{x})$ represents:

$$
f_k(\mathbf{x})
=
P(y = k \mid y \in \{k, \text{not-}k\}, \mathbf{x})
$$

These outputs do *not* sum to 1:

$$
\sum_{k=1}^K f_k(\mathbf{x}) \neq 1
$$

unlike softmax regression.


#### Decision Rule for Final Prediction

For a test sample $\mathbf{x}_{test}$:

$$
\mathbf{s} = [f_1(\mathbf{x}_{test}), f_2(\mathbf{x}_{test}), \dots, f_K(\mathbf{x}_{test})]^T
$$

Prediction is:

$$
\hat{y} = \arg\max_{k} f_k(\mathbf{x}_{test})
$$


#### Matrix Representation for Efficient Computation

Define:

$$
\mathbf{W} =
[\mathbf{w}_1 \mid \mathbf{w}_2 \mid \dots \mid \mathbf{w}_K]
\in \mathbb{R}^{d \times K}
$$

$$
\mathbf{b} = [b_1, b_2, \dots, b_K]^T
$$

For test data $\mathbf{X}_{test} \in \mathbb{R}^{m \times d}$:

$$
\mathbf{Z} =
\mathbf{X}_{test} \mathbf{W}
+
\mathbf{1}_m \mathbf{b}^T
$$

Apply sigmoid:

$$
\mathbf{S} = \sigma(\mathbf{Z})
$$


#### Handling Edge Cases in Representation

##### Ties in Prediction

if multiple classes have max score:
1. use smallest index
2. or compute margin difference
3. or random choice

##### Numerical Stability

$$
\sigma(z)=
\begin{cases}
\frac{1}{1 + e^{-z}}, & z \ge 0 \\
\frac{e^z}{1 + e^z}, & z < 0
\end{cases}
$$


#### Geometric Interpretation

Each classifier defines a hyperplane:

$$
\mathbf{w}_k^T \mathbf{x} + b_k = 0
$$


#### Comparison with Alternative Representations

Softmax regression models:

$$
P(y=k \mid \mathbf{x})=
\frac{e^{\mathbf{w}_k^T \mathbf{x}}}
{\sum_{j=1}^K e^{\mathbf{w}_j^T \mathbf{x}}}
$$

Advantages of OvA include parallelization, flexibility, and easy extension to new classes.


#### Summary of Model Parameters

- $K$ weight vectors $\mathbf{w}_k$  
- $K$ bias terms $b_k$  
- Total parameters:

$$
K(d + 1)
$$


### Mathematical Framework in All-Pairs (One-vs_One, OvO)

While OvA compares each class against all others, the all-pairs (one-vs-one, OvO) strategy builds a binary classifier **for every ordered pair of distinct classes**. This yields a more local set of decision boundaries and is especially common with margin-based models such as SVMs, but the same representation idea applies to logistic regression.

### Pairwise Binary Problems

For $K$ classes, we construct a binary classifier for each unordered pair $(k,\ell)$ with $k < \ell$. The total number of classifiers is:

$$
\frac{K(K-1)}{2}
$$

For each pair $(k,\ell)$, we restrict the training data to only those samples with $y_i \in \{k,\ell\}$ and relabel them:

$$
y_i^{(k,\ell)} =
\begin{cases}
1 & \text{if } y_i = k \\
0 & \text{if } y_i = \ell
\end{cases}
$$

Each pairwise classifier $(k,\ell)$ has its own parameters $(\mathbf{w}_{k,\ell}, b_{k,\ell})$ and outputs a score:

$$
f_{k,\ell}(\mathbf{x}) =
\sigma(\mathbf{w}_{k,\ell}^T \mathbf{x} + b_{k,\ell})
$$

We interpret $f_{k,\ell}(\mathbf{x})$ as the probability that $\mathbf{x}$ belongs to class $k$ rather than class $\ell$ in the local two-class problem.

### Decision Rule for Final Prediction (OvO)

Given a test sample $\mathbf{x}_{test}$, we let each pairwise classifier "vote" for its preferred class:

- If $f_{k,\ell}(\mathbf{x}_{test}) > 0.5$, the pair $(k,\ell)$ votes for class $k$.  
- Otherwise, the pair votes for class $\ell$.

We then count votes over all pairs:

- Let $v_c$ be the number of votes for class $c$.  
- The final prediction is:

$$
\hat{y} = \arg\max_{c \in \{1,\dots,K\}} v_c
$$

In case of ties, we can apply the same tie-breaking rules as in OvA (smallest index, margin-based, or random choice).

Compared to OvA, OvO uses more classifiers but each classifier only sees two classes, which can yield sharper local decision boundaries.

### Model Parameters

For OvA with logistic regression:

- $K$ weight vectors $\mathbf{w}_k$  
- $K$ bias terms $b_k$  
- Total parameters:
  $$
  K(d + 1)
  $$

For OvO with logistic regression:

- One classifier per pair $(k,\ell)$, so $\frac{K(K-1)}{2}$ classifiers in total.  
- Each classifier has its own $(\mathbf{w}_{k,\ell}, b_{k,\ell})$ parameters.

This completes the representation of both One-vs-All and All-Pairs multiclass schemes using a generic binary classifier such as logistic regression.

### OvA pseudo code (one-vs-all)

- Training:
  - For k = 1,…,K:
    - Define binary labels: y_bin[i] = 1 if y[i] == k, else 0
    - Train binary classifier f_k on (X, y_bin)

- Prediction for x:
  - Compute scores s_k = f_k(x) for all k
  - ŷ = argmax_k s_k


### OvO pseudo code (one-vs-one)

- Training:
  - For each class pair (k, ℓ) with k < ℓ:
    - Select samples with y ∈ {k, ℓ}
    - Define labels: y_pair[i] = 1 if y[i] == k, else 0
    - Train binary classifier f_{k,ℓ} on (X_pair, y_pair)

- Prediction for x:
  - Initialize votes[c] = 0 for all classes c
  - For each pair (k, ℓ):
    - If f_{k,ℓ}(x) > 0.5: votes[k] += 1
    - Else: votes[ℓ] += 1
  - ŷ = argmax_c votes[c]

---

## Loss

Multiclass classification with **logistic regression** relies on transforming the problem into multiple binary classification tasks. Two common strategies are **one-vs-all (OvA)** and **all-pairs (one-vs-one, OvO)**. Both approaches use the **binary cross-entropy loss** to measure the discrepancy between model predictions and true labels.

### One-vs-All Logistic Regression

In the **OvA** approach, a model trains **$K$** binary classifiers for a task with $K$ classes. Each classifier $k$ predicts whether a sample belongs to class $k$ versus all others. For an input $x_i$, the classifier outputs $\hat{y}_{ik} = \sigma(w_k^\top x_i)$

where $\sigma(\cdot)$ is the sigmoid function. The loss for classifier $k$ is:

$$
L_k = -\frac{1}{N} \sum_{i=1}^N 
\left[
y_{ik} \log(\hat{y}_{ik}) + (1 - y_{ik}) \log(1 - \hat{y}_{ik})
\right],
$$

where $y_{ik}=1$ if the true class is $k$, otherwise $0$. The overall loss is:

$$
L_{\text{OvA}} = \frac{1}{K} \sum_{k=1}^K L_k.
$$


**Pseudo-code for OvA training:**


- for each clas k:
    - Initialize weight vector $w_k$
- for each iteration:
    - Compute predictions: $\hat{y} = \sigma(X \, w_k)$
    - Compute gradients: $\nabla = X^{\top}(\hat{y} - y_k)$
    - Update weights: $w_k = w_k - \text{lr} \cdot \nabla$



### All-Pairs (One-vs-One) Logistic Regression

In the **OvO** method, a model trains a classifier for every pair of classes \((a, b)\). The number of classifiers is: $M = \binom{K}{2}$.

Each classifier is trained only on samples belonging to classes \(a\) and \(b\). The binary loss is:

$$
L_{a,b} = -\frac{1}{N_{a,b}} \sum_{i \in \{a,b\}}
\left[
y_i^{(a,b)} \log(\hat{y}_i^{(a,b)}) +
(1 - y_i^{(a,b)}) \log(1 - \hat{y}_i^{(a,b)})
\right].
$$

The conceptual overall loss is:

$$
L_{\text{OvO}} = \frac{1}{M} \sum_{a<b} L_{a,b}.
$$

**Pseudo-code for OvO training:**

- for each pair (a, b):
    - Extract subset of data $X_{a,b}$
    - Initialize weight vector $w_{a,b}$
- for each iteration:
    - Compute logits: $z = X_{a,b}$ , $w_{a,b}$
    - Compute predictions: $\hat{y} = \sigma(z)$
    - Compute gradients: $\nabla = X_{a,b}^T (\hat{y} - y_{a,b})$
    - Update weights: $w_{a,b} = w_{a,b} - \text{lr} \cdot \nabla$


Both OvA and OvO use **binary cross-entropy** as the metric that quantifies the error between predicted probabilities and true labels, providing a smooth, differentiable objective suitable for gradient-based optimization.

---
## Optimizer

Adam (Adaptive Moment Estimation) is an adaptive optimization algorithm that combines the advantages of momentum (to accelerate convergence) and adaptive learning rates (to handle sparse gradients). It maintains estimates of both the first-order moments (mean) and second-order moments (uncentered variance) of the gradients, with bias correction to address initializations near zero.

Compared to SGD and mini batch SGD, Adam is not sensitive to learning rate and easy to tune. Also, it converges very fast with maintaining high accuracy.

Also, in OvA (One vs All), each classifier's gradient is computed using all training samples, which often lead to imbalanced gradient distribution since more samples in the "negative" class. And Adam's adaptive learning rates and momentum can counteract this noise. Firstly, the second-moment estimate v reduces step sizes for parameters with noisy gradient, preventing overshooting. Secondly, the first moment estimate m smooths out noise via exponential averaging, stabilizing convergence.

### Hyperparameters

- $\alpha$: Learning rate 
- $\beta_1$: Exponential decay rate for the first-moment estimate 
- $\beta_2$: Exponential decay rate for the second-moment estimate 
- $\epsilon$: Small constant to avoid division by zero

### Mathematical Formulation

1. Initialize parameters and moment estmater:

- $\theta_0$: Initial guess for model parameters (random small values)
- $m_0 = 0$: Initial first-moment estimate (mean of gradients).
- $v_0 = 0$: Initial second-moment estimate (uncentered variance of gradients).

2. For each iteration t=1,2,... until convergence:

- Compute the gradient of the loss function with respect to $\theta$ using the current mini-batch of training data: $$g_t = {\nabla}_{\theta}L(\theta_{t-1})$$ where $L(\theta)$ is the Loss Function
- Update the first-moment estimate (exponentially weighted average of past gradients):$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$
- Update the second-moment estimate (exponentially weighted average of squared past gradients):$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$
- Apply bias correction to the moment estimates (to account for initializations near zero):$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \quad \text{(corrected first moment)}$ $\hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad \text{(corrected second moment)}$
- Update the model parameters using the corrected moments:$$\theta_t = \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

### pesudo code

Input: 
  - θ: Model parameters (W, b) to optimize
  - α: Learning rate
  - β₁, β₂: Decay rates
  - ε: Small constant
  - Training data: {(x₁, y₁), ..., (x_N, y_N)}
  - Batch size: B

Initialize:
  - m = 0  # First-moment estimate (same shape as θ)
  - v = 0  # Second-moment estimate (same shape as θ)
  - t = 0  # Iteration counter

While not converged:
  - t = t + 1
  - Mini-batch = random sample of B examples from training data
  - g = ∇_θ L(θ; Mini-batch) 
  
  - m = β₁ * m + (1 - β₁) * g
  - v = β₂ * v + (1 - β₂) * (g ⊙ g)  # ⊙ denotes element-wise multiplication
  

  - m̂ = m / (1 - β₁^t)
  - v̂ = v / (1 - β₂^t)
  

  - θ = θ - α * (m̂ / (sqrt(v̂) + ε))

Output: Optimized parameters θ